## Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
* Tracking agent behavior with logging, analytics and debugging
* Transforming prompts, tool selection and output formatting
* Adding retries, fallbacks and early termination logic
* Applying rate limits, guardrails and PII detection

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

### Summarization Middleware

Automatically Summarizes conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
* Long-running conversations that exceed context windows
* Multi-turn dialogues with extensive history
* Applications where preserving full conversation context matters

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [6]:
 ### Run with thread Id
config={"configurable":{"thread_id":"test-1"}}

In [7]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?"
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='6e750c8d-c7cd-4568-b4b5-c71dc936e4b5'), AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What is 2+2?"\n2.  **Identify Core Task:** This is a basic arithmetic question.\n3.  **Perform Calculation:** 2 + 2 = 4.\n4.  **Formulate Response:** State the answer clearly and concisely.\n5.  **Check for Accuracy:** 2+2 is universally accepted as 4 in standard arithmetic.\n6.  **Final Output:** "2 + 2 equals 4." (or simply "4")\n</think>\n\n2 + 2 equals 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 137, 'prompt_tokens': 17, 'total_tokens': 154, 'completion_time': 0.264999539, 'completion_tokens_details': None, 'prompt_time': 0.000215207, 'prompt_tokens_details': None, 'queue_time': 0.048567903, 'total_time': 0.265214746}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_fff3b79855', 'ser

## Token Size

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("tokens", 550),
            keep=("tokens",200),
        ),
        
    ]
)

config={"configurable":{"thread_id":"test-1"}}
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars


In [ ]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
    {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
    config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~7046 tokens, 6 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Role:** Context Extraction Assistant\n   - **Objective:** Extract the highest quality/most relevant context from the conversation history to overwrite it.\n   - **Required Structure:**\n     - ## SESSION INTENT\n     - ## SUMMARY\n     - ## ARTIFACTS\n     - ## NEXT STEPS\n   - **Input Messages:**\n     - Human: Previous summary (already structured) + new request "Find hotels in London" (implied from the history, actually the human message contains the previous AI output which was a summary, then AI called tool, tool returned results, AI presented results). Wait, let\'s look at the actual `<messages>` block carefully.\n     - `<message type="human">` contains a previous AI response that was already a summary. This is a bit meta. It seems the user pasted a previous turn that *was* a summary, then the AI calle

## Fraction

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

model = ChatGroq(
    model="qwen/qwen3.6-27b",
    profile={"max_input_tokens": 131072}
)

agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction", 0.005),
            keep=("fraction", 0.002),
        )
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

In [4]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    print(f"\n{city}")
    print(f"Messages: {len(response['messages'])}")

    for message in response["messages"]:
        if message.additional_kwargs.get("lc_source") == "summarization":
            print(">>> SUMMARIZATION OCCURRED")


Paris
Messages: 4

London
Messages: 8

Tokyo
Messages: 12

New York
Messages: 12
>>> SUMMARIZATION OCCURRED

Dubai
Messages: 12
>>> SUMMARIZATION OCCURRED

Singapore
Messages: 12
>>> SUMMARIZATION OCCURRED
